<a href="https://colab.research.google.com/github/gauravspyke/newrepo/blob/master/content/courses/deeplearning/notebooks/SiameseBERT_SemanticSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentence Embeddings using Siamese BERT-Networks
---
This Google Colab Notebook illustrates using the Sentence Transformer python library to quickly create BERT embeddings for sentences and perform fast semantic searches.

The Sentence Transformer library is available on [pypi](https://pypi.org/project/sentence-transformers/) and [github](https://github.com/UKPLab/sentence-transformers). The library implements code from the ACL 2019 paper entitled "[Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks](https://www.aclweb.org/anthology/D19-1410.pdf)" by Nils Reimers and Iryna Gurevych.


## Install Sentence Transformer Library

In [1]:
# Install the library using pip
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

## Load the BERT Model

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the BERT model. Various models trained on Natural Language Inference (NLI) https://github.com/UKPLab/sentence-transformers/blob/master/docs/pretrained-models/nli-models.md and
# Semantic Textual Similarity are available https://github.com/UKPLab/sentence-transformers/blob/master/docs/pretrained-models/sts-models.md

model = SentenceTransformer('bert-base-nli-mean-tokens')

100%|██████████| 405M/405M [00:51<00:00, 7.92MB/s]


In [35]:
import pandas as pd
df = pd.read_csv('text_comparisons_bert.csv')
df.head()




,Image,Component,Similarity_Score,Image_Caption,Image_Description,Image_Diagram_Type,Image_Combined_Text,Component_Text
0,contractor.png,A. Door Structure,0.210966,Figure 14 – Kit iced-up,This image shows a detailed diagram of an airc...,Principle Diagram,Figure 14 – Kit iced-up This image shows a det...,The door structure of the Emergency Exit Door ...
1,contractor.png,B. Stops,0.176960,Figure 14 – Kit iced-up,This image shows a detailed diagram of an airc...,Principle Diagram,Figure 14 – Kit iced-up This image shows a det...,The stops of the EED include two upper stops a...
2,contractor.png,C. Centering Devices,0.185685,Figure 14 – Kit iced-up,This image shows a detailed diagram of an airc...,Principle Diagram,Figure 14 – Kit iced-up This image shows a det...,The centering devices of the EED include the l...
3,contractor.png,D. Handle and Push Button,0.151491,Figure 14 – Kit iced-up,This image shows a detailed diagram of an airc...,Principle Diagram,Figure 14 – Kit iced-up This image shows a det...,The handle and push button of the EED are inte...
4,contractor.png,E. Latch Mechanism,0.248530,Figure 14 – Kit iced-up,This image shows a detailed diagram of an airc...,Principle Diagram,Figure 14 – Kit iced-up This image shows a det...,The latch mechanism of the EED is operated wit...


In [13]:
for index, row in df.iterrows():
  print(row['Image_Description'])

This image shows a detailed diagram of an aircraft fuel system component, specifically illustrating the parts and assembly of a fuel system kit in an iced-up condition. It highlights various components such as the piston, latch, handle, and rod body.
This image shows a detailed diagram of an aircraft fuel system component, specifically illustrating the parts and assembly of a fuel system kit in an iced-up condition. It highlights various components such as the piston, latch, handle, and rod body.
This image shows a detailed diagram of an aircraft fuel system component, specifically illustrating the parts and assembly of a fuel system kit in an iced-up condition. It highlights various components such as the piston, latch, handle, and rod body.
This image shows a detailed diagram of an aircraft fuel system component, specifically illustrating the parts and assembly of a fuel system kit in an iced-up condition. It highlights various components such as the piston, latch, handle, and rod bo

In [27]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


def compare(text1, text2 ):

# Load the model
  model = SentenceTransformer('all-MiniLM-L6-v2')
    # Sample image descriptions and function texts
  image_desc = ["This image shows a detailed diagram of an aircraft fuel system component, specifically illustrating the parts and assembly of a fuel system kit in an iced-up condition. It highlights various components such as the piston, latch, handle, and rod body."]
  function_text = ["The door structure of the Emergency Exit Door (EED) is composed of one-part machined in aluminum alloy strengthened by stiffeners. It provides the structural integrity and support for all other components of the EED. The door structure is designed to withstand the pressure difference between the inside and outside of the aircraft during flight."]

  # Embed texts — convert_to_tensor=False returns NumPy (safest for sklearn)
  emb1 = model.encode(text1, convert_to_tensor=False)  # shape: [1, 384]
  emb2 = model.encode(text2, convert_to_tensor=False)  # shape: [1, 384]

  # Convert to 2D numpy arrays if needed
  emb1 = np.array(emb1).reshape(1, -1)
  emb2 = np.array(emb2).reshape(1, -1)

  # Compute cosine similarity
  similarity = cosine_similarity(emb1, emb2)[0][0]
  #print(f"Similarity: {similarity:.4f}")
  return similarity

In [40]:
similarity_matrix = {}

for index, row in df.iterrows():
    #print(row)
    image = row['Image']
    component = row['Component']
    val = compare2(row['Image_Description'], row['Component_Text'])  # your similarity func

    if image not in similarity_matrix:
        similarity_matrix[image] = {}

    similarity_matrix[image][component] = val
    print(f"Score {val:.4f} for image {image} and section {component}")

result_df = pd.DataFrame.from_dict(similarity_matrix, orient='index')
result_df.fillna(0, inplace=True)  # optional: fill missing with 0


#for index, row in df.iterrows():
#  similarityVal  = compare1(row['Image_Description'], row['Component_Text'])
#  print(f"Score {similarityVal} for image {row['Image']} and section {row['Component']}")


Score 0.2551 for image contractor.png and section A. Door Structure
Score 0.2001 for image contractor.png and section B. Stops
Score 0.2032 for image contractor.png and section C. Centering Devices
Score 0.2540 for image contractor.png and section D. Handle and Push Button
Score 0.2757 for image contractor.png and section E. Latch Mechanism
Score 0.1631 for image contractor.png and section F. Door Seal
Score 0.2329 for image contractor.png and section G. Proximity Sensor and Sensor Target
Score 0.2335 for image contractor.png and section H. Window Brackets
Score 0.1824 for image contractor.png and section I. Quick Release Pin
Score 0.1484 for image contractor.png and section J. Contactor Fitting
Score 0.3663 for image door.png and section A. Door Structure
Score 0.2888 for image door.png and section B. Stops
Score 0.3165 for image door.png and section C. Centering Devices
Score 0.3571 for image door.png and section D. Handle and Push Button
Score 0.2942 for image door.png and section E

In [41]:
result_df.to_csv("image_component_similarity_tfidf.csv", index=True)


In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def compare2(text1, text2):

  # Sample texts
  #text1 = "The aircraft door must be checked before takeoff."
  #text2 = "Before flight, inspect the airplane door mechanism."

  # Create TF-IDF Vectorizer
  vectorizer = TfidfVectorizer()

  # Fit and transform the texts
  tfidf_matrix = vectorizer.fit_transform([text1, text2])

  # Compute cosine similarity
  similarity = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]

  #print(f"TF-IDF Cosine Similarity: {similarity:.4f}")
  return similarity


## Setup a Corpus

In [22]:
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity




def compare1(text1, text2 ):
# Load model and tokenizer
  model_name = "princeton-nlp/sup-simcse-roberta-base"
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModel.from_pretrained(model_name)

    # Function to get embedding from SimCSE model
  def get_simcse_embedding(text):
      inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
      with torch.no_grad():
          outputs = model(**inputs, return_dict=True)
      # Use the first token ([CLS] or <s>), i.e., outputs.last_hidden_state[:, 0, :]
      embeddings = outputs.pooler_output  # already pooled
      return embeddings.squeeze().numpy().reshape(1, -1)

  # Example texts
  #text1 = "The aircraft door must be checked before takeoff."
  #text2 = "Inspect the airplane door locking mechanism before flight."

  # Get embeddings
  emb1 = get_simcse_embedding(text1)
  emb2 = get_simcse_embedding(text2)

  # Compute cosine similarity
  similarity = cosine_similarity(emb1, emb2)[0][0]
  #print(f"SimCSE RoBERTa Cosine Similarity: {similarity:.4f}")
  return similarity


## Perform Semantic Search

In [ ]:
import scipy
#@title Sematic Search Form

# code adapted from https://github.com/UKPLab/sentence-transformers/blob/master/examples/application_semantic_search.py

query = 'Nobody has sane thoughts' #@param {type: 'string'}

queries = [query]
query_embeddings = model.encode(queries)

# Find the closest 5 sentences of the corpus for each query sentence based on cosine similarity
number_top_matches = 5 #@param {type: "number"}

print("Semantic Search Results")

for query, query_embedding in zip(queries, query_embeddings):
    distances = scipy.spatial.distance.cdist([query_embedding], sentence_embeddings, "cosine")[0]

    results = zip(range(len(distances)), distances)
    results = sorted(results, key=lambda x: x[1])

    print("\n\n======================\n\n")
    print("Query:", query)
    print("\nTop 5 most similar sentences in corpus:")

    for idx, distance in results[0:number_top_matches]:
        print(sentences[idx].strip(), "(Cosine Score: %.4f)" % (1-distance))

Semantic Search Results




Query: Nobody has sane thoughts

Top 5 most similar sentences in corpus:
Lack of saneness (Cosine Score: 0.8958)
Absence of sanity (Cosine Score: 0.8744)
A man is riding a horse. (Cosine Score: 0.1705)
